In [1]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

2026-05-11 18:32:48.523277: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778524368.733351      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778524368.790913      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778524369.311209      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778524369.311246      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778524369.311249      57 computation_placer.cc:177] computation placer alr

In [3]:
# 1. FIX RANDOMNESS FOR STABLE RESULTS
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [4]:
# ── 2. LOAD DATA ───────────────────────────────────────────────────────────
try:
    X_train = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/X_train.npy')
    X_test  = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/X_test.npy')
    y_train = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/y_train.npy')
    y_test  = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/y_test.npy')
    print(f"✓ Loaded. X_train: {X_train.shape}, y_train distribution: {np.bincount(y_train.astype(int))}")
except FileNotFoundError:
    print("Error: Check dataset paths.")

✓ Loaded. X_train: (40, 5, 5), y_train distribution: [14 26]


In [5]:
# ── 3. SCALE (fit on train only, transform both) ───────────────────────────
scaler = StandardScaler()
n_samples, n_steps, n_features = X_train.shape

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(X_train.shape)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(X_test.shape)

In [6]:
# ── 4. FIX IMBALANCE WITH SMOTE ───────────────────────────────────────────
# Flatten to 2D for SMOTE, then reshape back
X_train_flat = X_train_scaled.reshape(n_samples, -1)  # (samples, steps*features)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_flat, y_train)
X_train_bal = X_resampled.reshape(-1, n_steps, n_features)
print(f"✓ After SMOTE — X: {X_train_bal.shape}, y dist: {np.bincount(y_resampled.astype(int))}")

✓ After SMOTE — X: (52, 5, 5), y dist: [26 26]


In [7]:
# ── 5. CORRECT CLASS WEIGHTS (minority gets higher weight) ────────────────
classes = np.unique(y_resampled)
weights = compute_class_weight('balanced', classes=classes, y=y_resampled)
class_weight_dict = dict(zip(classes.astype(int), weights))
print(f"✓ Class weights: {class_weight_dict}")

✓ Class weights: {np.int64(0): np.float64(1.0), np.int64(1): np.float64(1.0)}


In [8]:
# ── 6. MODEL — tiny GRU matching data scarcity ────────────────────────────
# With <100 samples even after SMOTE, keep this SMALL
model = Sequential([
    Input(shape=(n_steps, n_features)),
    GRU(
        16,                                      # Small unit count
        activation='tanh',
        recurrent_dropout=0.2,                   # Dropout inside recurrent connections
        kernel_regularizer=regularizers.l2(0.02) # Stronger L2 for tiny dataset
    ),
    BatchNormalization(),
    Dropout(0.4),
    Dense(8, activation='relu', kernel_regularizer=regularizers.l2(0.02)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),  # Lower LR than your 0.005
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

I0000 00:00:1778524428.971817      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778524428.978430      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 16)             │         1,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313 (5.13 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 32 (128.00 B)

In [10]:
# ── 7. CALLBACKS ──────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    )
]

In [11]:
# ── 8. SINGLE TRAINING (your notebook had TWO — that was a bug) ───────────
history = model.fit(
    X_train_bal, y_resampled,
    epochs=300,
    batch_size=8,              # Slightly larger batch after SMOTE
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 121ms/step - accuracy: 0.4291 - loss: 1.6106 - val_accuracy: 0.0000e+00 - val_loss: 1.1884
Epoch 2/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5872 - loss: 1.3394 - val_accuracy: 0.0000e+00 - val_loss: 1.1706
Epoch 3/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6161 - loss: 1.3559 - val_accuracy: 0.0909 - val_loss: 1.1506
Epoch 4/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5067 - loss: 1.1574 - val_accuracy: 0.1818 - val_loss: 1.1321
Epoch 5/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6709 - loss: 1.3592 - val_accuracy: 0.0909 - val_loss: 1.1120
Epoch 6/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5653 - loss: 1.2862 - val_accuracy: 0.0000e+00 - val_loss: 1.0903
Epoch 7/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5113 - loss: 1.2975 - val_accuracy: 0.0000e+00 - val_loss: 1.0704
Epoch 8/300
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6622 - loss: 1.1881 - val_accuracy: 0

In [12]:
# ── 9. FIND OPTIMAL THRESHOLD ────────────────────────
from sklearn.metrics import f1_score as f1

y_val_prob = model.predict(X_train_bal)
best_thresh, best_f1 = 0.5, 0
for thresh in np.arange(0.3, 0.8, 0.05):
    preds = (y_val_prob > thresh).astype(int)
    score = f1(y_resampled, preds, zero_division=0)
    if score > best_f1:
        best_f1, best_thresh = score, thresh
print(f"✓ Best threshold: {best_thresh:.2f} (F1={best_f1:.4f})")

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 306ms/step
✓ Best threshold: 0.45 (F1=0.7541)


In [13]:
# ── 10. EVALUATE ON TEST ──────────────────────────────────────────────────
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > best_thresh).astype(int)

print("\n" + "="*40)
print(f"  GRU ACCURACY : {accuracy_score(y_test, y_pred):.4f}")
print(f"  GRU F1 SCORE : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print("\n  CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, zero_division=0))
print("  CONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))
print("="*40)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

  GRU ACCURACY : 0.6000
  GRU F1 SCORE : 0.6667

  CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         6
           1       0.50      1.00      0.67         4

    accuracy                           0.60        10
   macro avg       0.75      0.67      0.58        10
weighted avg       0.80      0.60      0.57        10

  CONFUSION MATRIX:
[[2 4]
 [0 4]]


In [16]:
import pandas as pd

# Save model
model.save('gru.keras')
print('✓ Model saved as gru.keras')

# Save training history as CSV
history_df = pd.DataFrame(history.history)
history_df.insert(0, 'epoch', range(1, len(history_df) + 1))
history_df.to_csv('gru.csv', index=False)
print(f'✓ Training history saved as gru.csv ({len(history_df)} epochs)')
print(history_df.tail())

✓ Model saved as gru.keras
✓ Training history saved as gru.csv (74 epochs)
    epoch  accuracy      loss  val_accuracy  val_loss
69     70  0.658537  0.750849      0.636364  0.928046
70     71  0.609756  0.764238      0.636364  0.922740
71     72  0.731707  0.805284      0.636364  0.917725
72     73  0.731707  0.730382      0.636364  0.912288
73     74  0.658537  0.881002      0.636364  0.911787
